# 04 — QK-Norm and grouped-query attention (M1.4)

The complete Qwen3 attention sublayer, built from the tested modules in
`qwen3_scratch.layers` (RMSNorm → QK-Norm, RoPE, `GroupedQueryAttention`).
Small synthetic tensors, CPU only, no downloads. Plots need `matplotlib`;
without it, text output is printed instead.

Write-ups: `docs/components/qk_norm.md`, `docs/components/attention.md`.

## 1. Why self-attention

A feed-forward layer sees one token at a time. To predict the next word, a token
must read *other* tokens: "it" needs the noun it refers to. Self-attention lets
every position compute a weighted average of information from the positions it
is allowed to see. The weights are computed from the content itself.

In [ ]:
import math

import torch

from qwen3_scratch.config import QWEN3_0_6B_BASE, TINY_QWEN3
from qwen3_scratch.layers import (
    GroupedQueryAttention,
    QKNorm,
    RotaryEmbedding,
    attention_softmax,
    bool_to_additive_mask,
    causal_mask,
    make_position_ids,
    query_to_kv_head,
    repeat_kv,
)

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

_ = torch.manual_seed(0)

## 2. Q, K, V and 3. attention scores

For hidden states $X\in\mathbb{R}^{T\times H}$: $Q=XW_q^\top$, $K=XW_k^\top$,
$V=XW_v^\top$, split into heads of size $D$. Scores
$S=QK^\top/\sqrt D$ (`[T, T]` per head), then 4. $P=\mathrm{softmax}(S+M)$ row-wise
and output $PV$.

Tiny example: 3 tokens, one head, $D=2$.

In [ ]:
q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [1.0, 1.0], [0.0, 2.0]])
v = torch.tensor([[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]])
scores = q @ k.T / math.sqrt(2)
allowed = causal_mask(torch.arange(3), torch.arange(3))
probs = attention_softmax(scores[None, None], allowed)[0, 0]
print("scores:\n", scores)
print("allowed (causal):\n", allowed)
print("probs (rows sum to 1):\n", probs)
print("output = probs @ v:\n", probs @ v)

## 5. Causal masking

A language model predicts token $t+1$ from tokens $\le t$, so position $i$ may only
attend to keys $j\le i$. The mask is applied to the **scores before softmax**:
masked scores become $-\infty$, so $e^{-\infty}=0$ and the remaining weights still
sum to 1. Masking *after* softmax would leave rows that no longer sum to 1.

We build masks from **absolute positions** (`k_pos <= q_pos`). That stays correct
when queries and keys have different lengths (cached decoding), where a plain
upper-triangular matrix would be wrong.

In [ ]:
T = 6
mask_full = causal_mask(torch.arange(T), torch.arange(T))
mask_decode = causal_mask(torch.tensor([4, 5]), torch.arange(T))  # 2 new tokens, 6 keys
print("decode-step mask [2, 6]:\n", mask_decode.int())
print("HF-style additive mask row 0:", bool_to_additive_mask(mask_full, torch.float32)[0].tolist())

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))
    for ax, m, title in [
        (axes[0], mask_full, "prefill: 6 queries × 6 keys"),
        (axes[1], mask_decode, "decode: queries at 4,5 × keys 0..5"),
    ]:
        ax.imshow(m.int().numpy(), cmap="Blues", vmin=0, vmax=1.4)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("key position j")
        ax.set_ylabel("query position")
        ax.set_xticks(range(m.shape[1]))
        ax.set_yticks(range(m.shape[0]))
    axes[1].set_yticklabels(["4", "5"])
    fig.suptitle("Causal mask: dark = may attend")
    plt.tight_layout()
    plt.show()

## 6–8. MHA, MQA, GQA

| variant | KV heads | query → KV mapping | K/V params per layer (H=1024, D=128) | KV cache per token per layer (bf16) |
|---|---|---|---|---|
| MHA | $N_{kv}=N_q=16$ | one-to-one | 4.19M | 8 KiB |
| **GQA** (Qwen3-0.6B) | $N_{kv}=8$ | groups of $g=N_q/N_{kv}=2$ | 2.10M | 4 KiB |
| MQA | $N_{kv}=1$ | all share one | 0.26M | 0.5 KiB |

Query heads keep their own projections, so all 16 attention patterns are still
different. Only keys and values are shared within a group. Query head $h$ uses KV
head $\lfloor h/g\rfloor$ (contiguous groups). A modulo mapping $h \bmod N_{kv}$
pairs different heads and silently breaks trained checkpoints.

In [ ]:
for name, nq, nkv in [("MHA", 8, 8), ("GQA", 8, 4), ("GQA", 8, 2), ("MQA", 8, 1)]:
    print(f"{name} {nq}/{nkv}: query head -> kv head {query_to_kv_head(nq, nkv)}")
print("modulo (wrong) 8/2:", [h % 2 for h in range(8)])
print("Qwen3-0.6B 16/8   :", query_to_kv_head(16, 8))

if plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.0), sharey=True)
    for ax, (name, nkv) in zip(
        axes,
        [("MHA (8 KV heads)", 8), ("GQA (2 KV heads)", 2), ("MQA (1 KV head)", 1)],
        strict=True,
    ):
        grid = torch.zeros(8, nkv)
        for h, kv in enumerate(query_to_kv_head(8, nkv)):
            grid[h, kv] = 1
        ax.imshow(grid.numpy(), cmap="Blues", vmin=0, vmax=1.4, aspect="auto")
        ax.set_title(name, fontsize=10)
        ax.set_xlabel("KV head")
        ax.set_xticks(range(nkv))
    axes[0].set_ylabel("query head")
    axes[0].set_yticks(range(8))
    fig.suptitle("Which KV head each of 8 query heads reads")
    plt.tight_layout()
    plt.show()

In [ ]:
k_small = torch.arange(2 * 3 * 1).float().view(1, 2, 3, 1)  # 2 KV heads, 3 tokens, D=1
print("KV heads:", k_small[0, :, :, 0].tolist())
print("repeat_kv(., 2) -> query heads:", repeat_kv(k_small, 2)[0, :, :, 0].tolist())

### KV-cache size estimates

In [ ]:
layers, d, nq = (
    QWEN3_0_6B_BASE.num_hidden_layers,
    QWEN3_0_6B_BASE.head_dim,
    QWEN3_0_6B_BASE.num_attention_heads,
)
contexts = [1024, 4096, 8192, 16384, 32768]
variants = {"MHA (16 KV)": 16, "GQA (8 KV, Qwen3)": 8, "MQA (1 KV)": 1}
gib = {
    name: [2 * nkv * d * 2 * layers * c / 2**30 for c in contexts] for name, nkv in variants.items()
}
print("bf16 KV cache, 28 layers, batch 1 (GiB):")
for name, vals in gib.items():
    print(
        f"  {name:18s}", "  ".join(f"{c:>6d}:{v:6.2f}" for c, v in zip(contexts, vals, strict=True))
    )

if plt is not None:
    colors = ["#2a78d6", "#eb6834", "#1baf7a"]
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for (name, vals), color in zip(gib.items(), colors, strict=True):
        ax.plot(contexts, vals, lw=2, marker="o", ms=5, color=color, label=name)
    ax.set_xlabel("context length (tokens)")
    ax.set_ylabel("KV cache (GiB)")
    ax.grid(alpha=0.25, lw=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False)
    ax.set_title("Qwen3-0.6B shapes: GQA halves the cache vs MHA")
    plt.show()

## 9. QK-Norm

Logits grow with $\|q\|\,\|k\|$. QK-Norm RMS-normalises every query and key head
vector (over $D$) with learnable scales $\gamma_q,\gamma_k$, so with unit scales
$|\hat q\cdot\hat k|/\sqrt D\le\sqrt D$ whatever the input scale. The $1/\sqrt D$
scaling is kept.

In [ ]:
d = 64
qkn = QKNorm(d)
for scale in (0.01, 1.0, 100.0):
    qs = scale * torch.randn(1, 2, 8, d)
    ks = scale * torch.randn(1, 2, 8, d)
    raw = (qs @ ks.mT / math.sqrt(d)).abs().max().item()
    qn, kn = qkn(qs, ks)
    normed = (qn @ kn.mT / math.sqrt(d)).abs().max().item()
    print(f"input scale {scale:6.2f}: max |logit| raw = {raw:10.3f}", end="   ")
    print(f"with QK-Norm = {normed:6.3f} (bound {math.sqrt(d):.1f})")

## 10–11. RoPE integration and the full forward pass

Order inside Qwen3 attention: projections → split heads → **QK-Norm** → **RoPE**
→ grouped KV → scaled scores → causal mask → float32 softmax → $PV$ → merge heads
→ `o_proj`. The trace below shows every intermediate shape for the tiny config
(`H=64`, `Nq=4`, `Nkv=2`, `D=32`; note `Nq·D = 128 ≠ H`).

In [ ]:
cfg = TINY_QWEN3
attn = GroupedQueryAttention.from_config(cfg)
rope = RotaryEmbedding.from_config(cfg)
x = torch.randn(2, 6, cfg.hidden_size)
cos, sin = rope(make_position_ids(6, offset=10))
y, tr = attn(x, cos, sin, return_trace=True)
for field in (
    "q_proj",
    "k_proj",
    "v_proj",
    "q_normed",
    "k_normed",
    "q_rope",
    "k_rope",
    "k_grouped",
    "v_grouped",
    "scores",
    "probs",
    "context",
    "output",
):
    print(f"{field:10s} {tuple(getattr(tr, field).shape)}")
print("rows sum to 1:", torch.allclose(tr.probs.sum(-1), torch.ones(2, 4, 6)))
print("future weights exactly 0:", bool((tr.probs.triu(1) == 0).all()))

if plt is not None:
    fig, ax = plt.subplots(figsize=(4, 3.4))
    im = ax.imshow(tr.probs[0, 0].detach().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_title("Attention probabilities, batch 0, head 0", fontsize=10)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    for i in range(6):
        for j in range(i + 1):
            val = tr.probs[0, 0, i, j].item()
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if val > 0.6 else "#1a1a19",
            )
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()

### Causal leakage check: change only the future

In [ ]:
prefix = torch.randn(1, 4, cfg.hidden_size)
a = torch.cat([prefix, torch.randn(1, 3, cfg.hidden_size)], dim=1)
b = torch.cat([prefix, 50 * torch.randn(1, 3, cfg.hidden_size)], dim=1)
cos7, sin7 = rope(make_position_ids(7))
out = attn(torch.cat([a, b]), cos7, sin7)
print("max diff on shared prefix positions 0..3:", (out[0, :4] - out[1, :4]).abs().max().item())
print("max diff on future positions 4..6       :", (out[0, 4:] - out[1, 4:]).abs().max().item())

## 12. Verification against Hugging Face

In [ ]:
try:
    import transformers
    from transformers.models.qwen3 import modeling_qwen3 as hf_mod
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf_cfg = transformers.Qwen3Config(
        hidden_size=cfg.hidden_size,
        num_attention_heads=cfg.num_attention_heads,
        num_key_value_heads=cfg.num_key_value_heads,
        head_dim=cfg.head_dim,
        rope_theta=cfg.rope_theta,
        rms_norm_eps=cfg.rms_norm_eps,
        attn_implementation="eager",
    )
    hf = hf_mod.Qwen3Attention(hf_cfg, layer_idx=0).eval()
    hf.load_state_dict(attn.state_dict())  # identical parameter names
    pos = make_position_ids(6, offset=10, batch_size=2)
    hcos, hsin = hf_mod.Qwen3RotaryEmbedding(hf_cfg)(x, pos)
    add_mask = bool_to_additive_mask(causal_mask(pos, pos), torch.float32)[:, None]
    with torch.no_grad():
        h_out, h_probs = hf(x, (hcos, hsin), add_mask)
        _, h_probs_nomask = hf(x, (hcos, hsin), None)
    print("output bit-identical:", torch.equal(h_out, y))
    print("probs  bit-identical:", torch.equal(h_probs, tr.probs))
    print("HF without a mask attends to the future:", bool((h_probs_nomask.triu(1) > 0).any()))

## 13. Gradient propagation

Gradients reach `o_proj`, `v_proj` (through $P$), `q_proj`/`k_proj` (through the
scores, RoPE and QK-Norm) and both norm scales. Because of the causal mask, the
loss at position $t$ produces no gradient for inputs at positions $> t$.

In [ ]:
attn.zero_grad()
xg = torch.randn(1, 5, cfg.hidden_size, requires_grad=True)
outg = attn(xg, *rope(make_position_ids(5)))
outg[0, 2].sum().backward()
print("grad norm per input position:", [round(v, 4) for v in xg.grad[0].norm(dim=-1).tolist()])
for name, p in attn.named_parameters():
    print(f"  {name:15s} grad norm {p.grad.norm().item():.4f}")

## 14. Numerical issues and common mistakes

- **Mask after softmax**, or a mask not broadcast over heads: rows no longer sum to
  1, or future tokens leak. The prefix test above catches leakage.
- **Upper-triangular masks for cached decoding**: with `Tq ≠ Tk`, build the mask
  from absolute positions.
- **Calling HF `Qwen3Attention` without a mask**: it is *not* causal by itself.
  `Qwen3Model` builds the mask.
- **Fully masked rows** (padding): `-inf` everywhere gives NaN. Ours returns zeros;
  HF's finite `finfo.min` gives a uniform row.
- **Modulo KV mapping** instead of contiguous groups.
- **`head_dim = H / Nq`**: Qwen3-0.6B has `Nq·D = 2048 ≠ H = 1024`.
- **QK-Norm after RoPE**, over `Nq·D` instead of per head, or applied to `V`.
- **Softmax in bf16**: HF computes it in float32 and casts back; we do the same.
  For float64 inputs we keep float64 (HF would downcast).
- **Scaling by `/ sqrt(D)` vs `* D**-0.5`**: equal mathematically, but they can differ
  in the last bit. We multiply, like HF, to stay bit-identical.

### Review questions
1. Why does GQA reduce KV-cache memory but not the number of attention patterns?
2. Which query heads of Qwen3-0.6B share KV head 5?
3. Why must the causal mask be applied before softmax?
4. Why is QK-Norm applied before RoPE and not after?
5. How large is the bf16 KV cache of Qwen3-0.6B for 8 sequences of 4,096 tokens?